# NII Rate/Volume Decomposition — Greek Systemic Banks

Splits the year-on-year change in net interest income (NII) into:
- **Rate (margin) effect**: change in NIM applied to last year's average assets, the part linked to ECB rate moves
- **Volume effect**: last year's NIM applied to the change in average assets
- **Mix effect**: the interaction of the two (small)

$$\Delta \text{NII} = \underbrace{\Delta \text{NIM} \times \bar{A}_{t-1}}_{\text{Rate}} + \underbrace{\text{NIM}_{t-1} \times \Delta \bar{A}}_{\text{Volume}} + \underbrace{\Delta \text{NIM} \times \Delta \bar{A}}_{\text{Mix}}$$

where $\text{NIM} = \text{NII} / \bar{A}$ and $\bar{A}$ is average total assets, both on the same perimeter (the NIM in `kpis_final.csv`, columns `nim_nii` and `nim_avg_assets`).

**Same perimeter.** Alpha sold Alpha Bank Romania and is selling Alpha Life (Project Unicorn), and Eurobank sold Eurobank Direktna in Serbia. Their income statements leave these businesses out, while their balance sheets held them for a while. For 2022–2023 Alpha's NII therefore includes the businesses' net interest income from the discontinued-operations note, because the reports do not give their balances before end-2023; for 2024 their assets held for sale are taken out instead. When a business leaves the basis (Alpha's Romanian bank and Alpha Life in 2024, NBG's insurer in 2023), its NII of the previous year is taken out of the volume effect, so the volume effect shows the growth of the businesses that stay. The walk explains the reported change in NII: Rate + Volume + Mix + **Sold** = reported ΔNII, where Sold is what is left, the change in NII of the businesses being sold that the margins include (Alpha 2022→2023: −€12m; zero elsewhere).

**Rate context:** ECB deposit facility rate, yearly average (`data/manual/ecb_deposit_rate.csv`).
**Basis:** 2022 figures are the restated comparatives of the FY2023 reports, so the reported NII compares like with like (for example Alpha's Romanian business is a discontinued operation in all three years).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(kpis)} bank-year rows.')

Loaded 12 bank-year rows.


In [2]:
# ── ECB deposit facility rate (yearly average) ───────────────────────────────
ecb = pd.read_csv(DATA_DIR.parent / 'manual' / 'ecb_deposit_rate.csv')
ecb_rate = dict(zip(ecb['year'], ecb['dfr_avg']))
source = ecb['source'].iloc[0]
print('ECB deposit facility rate, yearly average (%):', ecb_rate)

ECB deposit facility rate, yearly average (%): {2022: 0.08, 2023: 3.31, 2024: 3.73}


In [3]:
# ── NIM × average assets decomposition ───────────────────────────────────────
# Same perimeter as the NIM in kpis_final.csv (see the note above)
df = kpis[['bank','year','nii','nim_nii','nim_avg_assets']].copy()
df['nim_exact'] = df['nim_nii'] / df['nim_avg_assets']   # unrounded, so the walk adds up exactly
df['nii_added'] = df['nim_nii'] - df['nii']               # discontinued-operations NII added back

transitions = [(2023, 2022), (2024, 2023)]
rows = []
for bank in BANKS:
    bdf = df[df['bank'] == bank].set_index('year')
    for yr, yp in transitions:
        nim_t, nim_p = bdf.loc[yr, 'nim_exact'], bdf.loc[yp, 'nim_exact']
        a_t, a_p     = bdf.loc[yr, 'nim_avg_assets'], bdf.loc[yp, 'nim_avg_assets']
        nii_t, nii_p = bdf.loc[yr, 'nim_nii'], bdf.loc[yp, 'nim_nii']
        d_nim, d_a = nim_t - nim_p, a_t - a_p
        # When a business leaves the basis (its NII was added back last year, not this year),
        # its NII is taken out of the volume effect, which then shows the businesses that stay.
        add_t, add_p = bdf.loc[yr, 'nii_added'], bdf.loc[yp, 'nii_added']
        leaving = add_p if (add_p > 0 and add_t == 0) else 0.0
        delta_rep = bdf.loc[yr, 'nii'] - bdf.loc[yp, 'nii']
        rate, volume, mix = d_nim * a_p, nim_p * d_a + leaving, d_nim * d_a
        rows.append({
            'bank': bank, 'year': yr,
            'nii_prev': bdf.loc[yp, 'nii'], 'nii_t': bdf.loc[yr, 'nii'], 'delta_nii': delta_rep,
            'rate_effect': rate,
            'volume_effect': volume,
            'mix_effect': mix,
            'sold_effect': delta_rep - rate - volume - mix,
            'd_nim_pp': d_nim * 100,
            'd_assets_pct': d_a / a_p * 100,
        })

decom = pd.DataFrame(rows)
decom['check_sum'] = decom[['rate_effect','volume_effect','mix_effect','sold_effect']].sum(axis=1)
decom['mismatch']  = (decom['check_sum'] - decom['delta_nii']).abs()
decom['rate_share'] = decom['rate_effect'] / decom['delta_nii'] * 100

print('Decomposition (€ million):')
print(decom[['bank','year','delta_nii','rate_effect','volume_effect','mix_effect','sold_effect']].round(1).to_string(index=False))

Decomposition (€ million):
        bank  year  delta_nii  rate_effect  volume_effect  mix_effect  sold_effect
    Eurobank  2023      694.0        635.7           40.8        17.5          0.0
    Eurobank  2024      333.0         26.8          302.5         3.7          0.0
  Alpha Bank  2023      485.2        509.0           -8.2        -3.1        -12.5
  Alpha Bank  2024      -14.0        -25.6            9.4         2.2          0.0
Piraeus Bank  2023      650.0        692.8          -28.3       -14.5         -0.0
Piraeus Bank  2024       85.0         13.0           71.6         0.5          0.0
         NBG  2023      894.0       1024.9          -71.6       -59.3         -0.0
         NBG  2024       93.0        142.7          -46.8        -2.9         -0.0


In [4]:
# ── Chart 1: Waterfall Charts per Bank ────────────────────────────────────────
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    vertical_spacing=0.18, horizontal_spacing=0.10,
)

BAR_COLORS = {
    'rate_effect':   '#00CC96',
    'volume_effect': '#636EFA',
    'mix_effect':    '#AB63FA',
    'sold_effect': '#FFA15A',
}
EFFECT_LABELS = {
    'rate_effect': 'Rate/Spread (ΔNIM × prior assets)',
    'volume_effect': 'Volume (prior NIM × ΔAssets)',
    'mix_effect': 'Mix (ΔNIM × ΔAssets)',
    'sold_effect': 'Businesses sold (Alpha)',
}
effects = ['rate_effect', 'volume_effect', 'mix_effect', 'sold_effect']

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    bdf = decom[decom['bank'] == bank].sort_values('year')
    x_labels = [f"{int(r['year'])-1}→{int(r['year'])}" for _, r in bdf.iterrows()]

    for effect in effects:
        y_vals = bdf[effect].values
        fig1.add_trace(
            go.Bar(
                x=x_labels, y=y_vals,
                name=EFFECT_LABELS[effect],
                marker_color=BAR_COLORS[effect],
                showlegend=(bi == 0), legendgroup=effect,
                text=[f'{v:+.0f}' for v in y_vals], textposition='outside',
            ),
            row=row + 1, col=col + 1,
        )

    fig1.add_trace(
        go.Scatter(
            x=x_labels, y=bdf['delta_nii'].values,
            mode='markers+text',
            marker=dict(symbol='diamond', size=12, color='white'),
            text=[f'Δ{v:+.0f}m' for v in bdf['delta_nii'].values],
            textposition='top center',
            name='Total ΔNII', showlegend=(bi == 0), legendgroup='delta',
        ),
        row=row + 1, col=col + 1,
    )

fig1.update_layout(
    title_text='<b>NII Rate vs Volume Waterfall</b> | ΔNIM × Assets decomposition (€ million)',
    title_font_size=16, template='plotly_dark', height=620, barmode='relative',
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center', font_size=11),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig1.show()

In [5]:
# ── Chart 3: ECB rate context ─────────────────────────────────────────────────
fig3 = make_subplots(rows=1, cols=2, subplot_titles=['ECB deposit rate (yearly average)', 'Sector NII vs ECB rate'])

fig3.add_trace(
    go.Bar(x=list(ecb_rate.keys()), y=list(ecb_rate.values()),
           marker_color=['#636EFA','#EF553B','#00CC96'],
           text=[f'{v:.2f}%' for v in ecb_rate.values()], textposition='outside'),
    row=1, col=1,
)

sector_nii = kpis.groupby('year')['nii'].sum().reset_index()
fig3.add_trace(
    go.Scatter(x=[ecb_rate[y] for y in sector_nii['year']], y=sector_nii['nii'].values,
               mode='markers+text',
               marker=dict(size=14, color=['#636EFA','#EF553B','#00CC96']),
               text=[str(y) for y in sector_nii['year']],
               textposition='top center'),
    row=1, col=2,
)

fig3.update_layout(
    title_text='<b>ECB Rate Context</b> | Sector NII against the ECB deposit rate',
    template='plotly_dark', height=350, showlegend=False,
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig3.update_xaxes(title_text='Year', row=1, col=1)
fig3.update_yaxes(title_text='ECB deposit rate (%)', row=1, col=1)
fig3.update_xaxes(title_text='ECB deposit rate (%)', row=1, col=2)
fig3.update_yaxes(title_text='Sector NII (€m)', row=1, col=2)
fig3.show()

In [6]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(decom) == 8, f'Expected 8 rows (4 banks × 2 transitions), got {len(decom)}'
max_mismatch = decom['mismatch'].max()
assert max_mismatch < 0.5, f'Decomposition mismatch: {max_mismatch:.4f}m'
# Only Alpha 2022→2023 carries a sold-businesses term (their own NII growth, about €12m)
sold = decom.set_index(['bank', 'year'])['sold_effect']
assert abs(sold.drop(('Alpha Bank', 2023))).max() < 0.5 and abs(sold[('Alpha Bank', 2023)]) < 15

r23 = decom[decom['year'] == 2023]
assert (r23['rate_effect'] > 0).all(), 'Expected a positive rate effect for every bank in 2023'

print('All checks passed.')
print(f'   Max decomposition mismatch: {max_mismatch:.6f} €m')
print(f'   2022→2023: sector ΔNII €{r23["delta_nii"].sum():,.0f}m, of which rate effect '
      f'€{r23["rate_effect"].sum():,.0f}m ({r23["rate_effect"].sum()/r23["delta_nii"].sum()*100:.0f}%)')
r24 = decom[decom['year'] == 2024]
print(f'   2023→2024: sector ΔNII €{r24["delta_nii"].sum():,.0f}m: rate effect €{r24["rate_effect"].sum():,.0f}m, '
      f'volume effect €{r24["volume_effect"].sum():,.0f}m, mix €{r24["mix_effect"].sum():,.0f}m')
print(f'   ECB rate source: {source}')

All checks passed.
   Max decomposition mismatch: 0.000000 €m
   2022→2023: sector ΔNII €2,723m, of which rate effect €2,862m (105%)
   2023→2024: sector ΔNII €497m: rate effect €157m, volume effect €337m, mix €3m
   ECB rate source: ECB key interest rates (ecb.europa.eu)
